In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import amplitude_to_db, db_to_amplitude

In [ ]:
def variable_feedback_delay(x: np.ndarray, M: np.ndarray, g: float = 1.0) -> np.ndarray:
    """A feedback delay line with a time-varying, fractional delay M[n] >= 1:
    y[n] = x[n] + g * y[n - M[n]]."""
    L = int(np.ceil(M.max())) + 2     # a delay line long enough for the longest delay
    buffer = np.zeros(L)
    i = 0                             # where y[n] will be written
    y = np.zeros(len(x))
    for n in range(len(x)):
        k = int(M[n])                 # whole samples of delay...
        a = M[n] - k                  # ...and the fraction left over
        # interpolate between y[n - k] and y[n - k - 1], like a wavetable (Chapter 3)
        delayed = (1 - a) * buffer[(i - k) % L] + a * buffer[(i - k - 1) % L]
        y[n] = x[n] + g * delayed
        buffer[i] = y[n]
        i = (i + 1) % L
    return y


f_s = 44100
noise = np.random.uniform(-1.0, 1.0, 3 * f_s)
M = np.linspace(f_s / 110, f_s / 440, len(noise))   # sweep the comb's teeth up two octaves
y = variable_feedback_delay(noise, M, g=0.95)
pq.play(pq.Audio(noise, f_s).normalize(peak_dbfs=-30.0))
pq.play(pq.Audio(y, f_s), normalize=True)